# MML Chapter 4 — Matrix Decompositions

> Decompositions break matrices into structured factors — diagonal, orthogonal, triangular.  
> Each factorization reveals a different layer of meaning: geometry of the transformation,  
> energy distribution, compressed rank.

**Sections:**
1. Determinant — compute 2×2 and 3×3; verify det(AB) = det(A)det(B)
2. Eigenvalues & Eigenvectors — compute, verify Ax = λx, plot eigenvectors on transformed unit circle
3. Cholesky Decomposition — decompose an SPD matrix; sample from a multivariate Gaussian
4. Eigendecomposition — A = PDP⁻¹; verify; compute A¹⁰ via eigendecomp vs direct
5. SVD — compute, verify A = UΣVᵀ; geometric interpretation (unit circle → ellipse)
6. Low-Rank Approximation — Frobenius error vs rank on a 10×10 random matrix
7. Image Compression Demo — rank-k reconstructions of a synthetic image

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch

rng = np.random.default_rng(42)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'font.family': 'sans-serif',
})
print('Imports done.')

---
## 1. Determinant

The determinant measures the **signed volume** of the parallelepiped formed by the columns of a matrix.

Key formula for 2×2:
$$\det\begin{pmatrix}a & b \\ c & d\end{pmatrix} = ad - bc$$

Key property: $\det(AB) = \det(A)\,\det(B)$

In [ ]:
# --- 2×2 determinant ---
A2 = np.array([[3, 1],
               [2, 4]], dtype=float)

det_manual = A2[0, 0] * A2[1, 1] - A2[0, 1] * A2[1, 0]  # ad - bc
det_numpy  = np.linalg.det(A2)

print('2×2 matrix:')
print(A2)
print(f'  det (manual ad-bc) = {det_manual}')
print(f'  det (numpy)        = {det_numpy:.6f}')

# --- 3×3 determinant via Laplace expansion along first row ---
A3 = np.array([[1, 2, 3],
               [4, 0, 6],
               [7, 8, 9]], dtype=float)

def det3_laplace(M):
    """Cofactor expansion along row 0."""
    d = 0
    for j in range(3):
        # Minor: delete row 0, col j
        minor = np.delete(np.delete(M, 0, axis=0), j, axis=1)
        cofactor = ((-1) ** j) * (minor[0,0]*minor[1,1] - minor[0,1]*minor[1,0])
        d += M[0, j] * cofactor
    return d

det3_manual = det3_laplace(A3)
det3_numpy  = np.linalg.det(A3)

print('\n3×3 matrix:')
print(A3)
print(f'  det (Laplace expansion) = {det3_manual}')
print(f'  det (numpy)             = {det3_numpy:.6f}')

In [ ]:
# --- Verify det(AB) = det(A) * det(B) ---
B2 = np.array([[2, -1],
               [0,  3]], dtype=float)

AB = A2 @ B2
det_AB    = np.linalg.det(AB)
det_A_det_B = np.linalg.det(A2) * np.linalg.det(B2)

print('Verifying det(AB) = det(A) * det(B)')
print(f'  det(A)      = {np.linalg.det(A2):.6f}')
print(f'  det(B)      = {np.linalg.det(B2):.6f}')
print(f'  det(A)*det(B) = {det_A_det_B:.6f}')
print(f'  det(AB)       = {det_AB:.6f}')
print(f'  Match: {np.isclose(det_AB, det_A_det_B)}')

---
## 2. Eigenvalues and Eigenvectors

The eigenvector equation:
$$A\mathbf{x} = \lambda\mathbf{x} \quad (\mathbf{x} \neq 0)$$

The matrix $A$ stretches (or flips) $\mathbf{x}$ by factor $\lambda$ — direction preserved.

**Spectral theorem:** If $A = A^\top$, all eigenvalues are real and eigenvectors are orthogonal.

In [ ]:
# --- Symmetric 2×2: compute eigenvalues and eigenvectors ---
S = np.array([[3, 1],
              [1, 2]], dtype=float)  # symmetric

eigenvalues, eigenvectors = np.linalg.eig(S)
# Sort by eigenvalue magnitude (descending)
idx = np.argsort(eigenvalues)[::-1]
eigenvalues  = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

print('Matrix S:')
print(S)
print(f'\nEigenvalues:  λ₁ = {eigenvalues[0]:.4f},  λ₂ = {eigenvalues[1]:.4f}')
print(f'Eigenvectors (columns):')
print(eigenvectors)

# --- Verify Ax = λx for each eigenpair ---
print('\nVerification: Ax - λx should be ~zero')
for i in range(2):
    lam = eigenvalues[i]
    x   = eigenvectors[:, i]
    residual = S @ x - lam * x
    print(f'  Eigenpair {i+1}: residual = {residual}   (close to zero: {np.allclose(residual, 0)})')

In [ ]:
# --- Plot eigenvectors on top of transformed unit circle ---
theta = np.linspace(0, 2 * np.pi, 300)
unit_circle = np.array([np.cos(theta), np.sin(theta)])  # (2, 300)

# Apply S to each point on the unit circle
transformed = S @ unit_circle  # (2, 300)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

# Left: unit circle with eigenvectors
ax = axes[0]
ax.plot(unit_circle[0], unit_circle[1], 'steelblue', lw=1.5, label='Unit circle')
colors = ['#e74c3c', '#2ecc71']
for i in range(2):
    v = eigenvectors[:, i]
    ax.annotate('', xy=v, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=colors[i], lw=2))
    ax.text(v[0] * 1.15, v[1] * 1.15, f'$v_{i+1}$', color=colors[i], fontsize=12)
ax.set_aspect('equal')
ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)
ax.set_title('Unit circle + eigenvectors of S')
ax.set_xlim(-1.6, 1.6); ax.set_ylim(-1.6, 1.6)

# Right: transformed ellipse with scaled eigenvectors
ax = axes[1]
ax.plot(transformed[0], transformed[1], 'coral', lw=1.5, label='S · unit circle')
for i in range(2):
    v = eigenvectors[:, i] * eigenvalues[i]  # scaled by eigenvalue
    ax.annotate('', xy=v, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=colors[i], lw=2))
    ax.text(v[0] * 1.1, v[1] * 1.1, f'$\\lambda_{i+1}v_{i+1}$', color=colors[i], fontsize=11)
ax.set_aspect('equal')
ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)
ax.set_title('Transformed ellipse (S·unit_circle) + scaled eigenvectors')
ax.set_xlim(-4.5, 4.5); ax.set_ylim(-4.5, 4.5)

plt.suptitle('Eigendecomposition: eigenvectors align with ellipse axes', fontsize=13)
plt.tight_layout()
plt.show()

---
## 3. Cholesky Decomposition

For a **symmetric positive definite (SPD)** matrix $A$:
$$A = LL^\top$$
where $L$ is lower triangular with positive diagonal entries.

**ML application:** Sampling $\mathbf{x} \sim \mathcal{N}(\boldsymbol{\mu}, \Sigma)$:
$$\mathbf{x} = \boldsymbol{\mu} + L\mathbf{z}, \quad \mathbf{z} \sim \mathcal{N}(0, I)$$

In [ ]:
# --- Construct an SPD matrix and perform Cholesky ---
# SPD = M^T M + epsilon*I guarantees positive definiteness
M = np.array([[2, 1],
              [1, 3]], dtype=float)
# M itself is already SPD (eigenvalues positive, symmetric)

L = np.linalg.cholesky(M)  # lower triangular factor

print('SPD matrix M:')
print(M)
print('\nCholesky factor L (lower triangular):')
print(np.round(L, 6))
print('\nVerification: L @ L.T:')
print(np.round(L @ L.T, 10))
print(f'\nL @ L.T == M: {np.allclose(L @ L.T, M)}')
print(f'Diagonal of L (must be positive): {np.diag(L)}')

In [ ]:
# --- Sample from multivariate Gaussian using Cholesky ---
mu    = np.array([1.0, -0.5])   # mean
Sigma = np.array([[2.0, 1.2],
                  [1.2, 1.5]])  # covariance matrix (SPD)

L_sigma = np.linalg.cholesky(Sigma)

n_samples = 500
z = rng.standard_normal((2, n_samples))   # z ~ N(0, I)
x = mu[:, None] + L_sigma @ z             # x = mu + L*z

# Analytical: draw directly (for comparison)
x_direct = rng.multivariate_normal(mu, Sigma, n_samples).T

# Estimate empirical covariance
emp_cov = np.cov(x)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

for ax, samples, title in zip(axes,
                               [x, x_direct],
                               ['Cholesky samples (x = μ + Lz)',
                                'Direct numpy samples']):
    ax.scatter(samples[0], samples[1], alpha=0.35, s=12, color='steelblue')
    ax.scatter(*mu, color='crimson', s=80, zorder=5, label='Mean μ')
    ax.set_aspect('equal')
    ax.set_title(title)
    ax.set_xlabel('x₁'); ax.set_ylabel('x₂')
    ax.legend()

print(f'True covariance Σ:\n{Sigma}')
print(f'\nEmpirical covariance (Cholesky samples, n={n_samples}):\n{np.round(emp_cov, 3)}')

plt.suptitle(f'500 samples from N(μ, Σ) via Cholesky decomposition', fontsize=13)
plt.tight_layout()
plt.show()

---
## 4. Eigendecomposition

A square non-defective matrix $A$ can be factorized as:
$$A = PDP^{-1}$$
where $D = \text{diag}(\lambda_1, \ldots, \lambda_n)$ and $P$ has eigenvectors as columns.

**Powers made easy:**
$$A^k = PD^kP^{-1}$$
because $D^k = \text{diag}(\lambda_1^k, \ldots, \lambda_n^k)$.

In [ ]:
# --- Eigendecomposition A = P D P^{-1} ---
A = np.array([[4, 1],
              [2, 3]], dtype=float)

eigenvalues_A, P = np.linalg.eig(A)
D_diag = np.diag(eigenvalues_A)
P_inv  = np.linalg.inv(P)

A_reconstructed = P @ D_diag @ P_inv

print('A:')
print(A)
print(f'\nEigenvalues: {eigenvalues_A}')
print('\nP (eigenvector columns):')
print(np.round(P, 6))
print('\nD (diagonal of eigenvalues):')
print(np.round(D_diag, 6))
print('\nReconstructed P D P⁻¹:')
print(np.round(A_reconstructed, 10))
print(f'\nA == P D P⁻¹: {np.allclose(A, A_reconstructed)}')

In [ ]:
# --- Compute A^10 via eigendecomposition vs numpy direct ---
k = 10

# Method 1: eigendecomposition  A^k = P D^k P^{-1}
D_k      = np.diag(eigenvalues_A ** k)
A_k_eig  = P @ D_k @ P_inv

# Method 2: repeated matrix multiplication (numpy matrix power)
A_k_direct = np.linalg.matrix_power(A.astype(int), k).astype(float)

print(f'A^{k} via eigendecomposition:')
print(np.round(A_k_eig, 4))

print(f'\nA^{k} via numpy matrix_power:')
print(A_k_direct)

print(f'\nResults match: {np.allclose(A_k_eig, A_k_direct, atol=1e-3)}')

# Show relative error
rel_err = np.max(np.abs(A_k_eig - A_k_direct)) / np.max(np.abs(A_k_direct))
print(f'Max relative error: {rel_err:.2e}')

---
## 5. Singular Value Decomposition (SVD)

The universal factorization — works for **any** matrix $A \in \mathbb{R}^{m \times n}$:
$$A = U\Sigma V^\top$$

- $U \in \mathbb{R}^{m \times m}$: orthogonal (left singular vectors)
- $\Sigma \in \mathbb{R}^{m \times n}$: diagonal-ish (singular values $\sigma_1 \geq \sigma_2 \geq \cdots \geq 0$)
- $V \in \mathbb{R}^{n \times n}$: orthogonal (right singular vectors)

**Geometric reading:** Vᵀ rotates → Σ scales → U rotates.

In [ ]:
# --- Compute SVD and verify A = U Σ Vᵀ ---
A_rect = np.array([[3, 2, 2],
                   [2, 3, -2]], dtype=float)  # 2×3 matrix

U, s, Vt = np.linalg.svd(A_rect, full_matrices=True)

# Build Sigma matrix (same shape as A_rect)
Sigma_full = np.zeros_like(A_rect)
Sigma_full[:len(s), :len(s)] = np.diag(s)

A_reconstructed = U @ Sigma_full @ Vt

print('A (2×3):')
print(A_rect)
print(f'\nU (2×2, orthogonal):\n{np.round(U, 6)}')
print(f'\nSingular values σ: {np.round(s, 6)}')
print(f'\nΣ (2×3):\n{np.round(Sigma_full, 6)}')
print(f'\nVᵀ (3×3):\n{np.round(Vt, 6)}')
print(f'\nReconstructed UΣVᵀ:\n{np.round(A_reconstructed, 10)}')
print(f'\nA == UΣVᵀ: {np.allclose(A_rect, A_reconstructed)}')

In [ ]:
# --- Verify U and V are orthogonal ---
print('U is orthogonal (UᵀU = I):', np.allclose(U.T @ U, np.eye(U.shape[0])))
V = Vt.T
print('V is orthogonal (VᵀV = I):', np.allclose(Vt @ V, np.eye(V.shape[0])))
print(f'rank(A) = number of nonzero singular values = {np.sum(s > 1e-10)}')

In [ ]:
# --- Geometric interpretation: unit circle → ellipse via 2×2 SVD ---
B = np.array([[3, 1],
              [1, 2]], dtype=float)

U_b, s_b, Vt_b = np.linalg.svd(B)
V_b = Vt_b.T

# Unit circle
theta = np.linspace(0, 2 * np.pi, 400)
circle = np.array([np.cos(theta), np.sin(theta)])  # (2, 400)

# B applied to unit circle
ellipse = B @ circle

# The axes of the ellipse are U columns, scaled by singular values
sigma1_vec = s_b[0] * U_b[:, 0]
sigma2_vec = s_b[1] * U_b[:, 1]

# Right singular vectors (input directions mapped to ellipse axes)
v1 = V_b[:, 0]
v2 = V_b[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left panel: unit circle with right singular vectors
ax = axes[0]
ax.plot(circle[0], circle[1], 'steelblue', lw=1.5)
for vec, color, label in [(v1, '#e74c3c', '$v_1$'), (v2, '#2ecc71', '$v_2$')]:
    ax.annotate('', xy=vec, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=color, lw=2.5))
    ax.text(vec[0] * 1.2, vec[1] * 1.2, label, color=color, fontsize=13)
ax.set_aspect('equal')
ax.set_xlim(-1.6, 1.6); ax.set_ylim(-1.6, 1.6)
ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5)
ax.set_title('Domain: unit circle\n(right singular vectors $v_1, v_2$)')

# Right panel: ellipse with left singular vectors scaled by σ
ax = axes[1]
ax.plot(ellipse[0], ellipse[1], 'coral', lw=1.5)
for vec, color, label in [(sigma1_vec, '#e74c3c', f'$\\sigma_1 u_1$ ({s_b[0]:.2f})'),
                           (sigma2_vec, '#2ecc71', f'$\\sigma_2 u_2$ ({s_b[1]:.2f})')]:
    ax.annotate('', xy=vec, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=color, lw=2.5))
    ax.text(vec[0] * 1.12, vec[1] * 1.12, label, color=color, fontsize=11)
ax.set_aspect('equal')
ax.set_xlim(-4.5, 4.5); ax.set_ylim(-4.5, 4.5)
ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5)
ax.set_title('Codomain: B · unit circle (ellipse)\n(left singular vectors $\\sigma_i u_i$)')

plt.suptitle('SVD Geometry: Vᵀ rotates → Σ scales → U rotates', fontsize=13)
plt.tight_layout()
plt.show()

print(f'Singular values: σ₁ = {s_b[0]:.4f}, σ₂ = {s_b[1]:.4f}')
print(f'Ellipse semi-axes have lengths σ₁ and σ₂')

---
## 6. Low-Rank Approximation

**Eckart-Young theorem (1936):** The best rank-$k$ approximation to $A$ (in both spectral and Frobenius norms) is:
$$\hat{A}^{(k)} = \sum_{i=1}^{k} \sigma_i \mathbf{u}_i \mathbf{v}_i^\top$$

Approximation error: $\|A - \hat{A}^{(k)}\|_F = \sqrt{\sigma_{k+1}^2 + \cdots + \sigma_r^2}$

In [ ]:
# --- Low-rank approximation of a 10×10 random matrix ---
rng2 = np.random.default_rng(7)
R = rng2.standard_normal((10, 10))  # random 10×10 (rank 10)

U_r, s_r, Vt_r = np.linalg.svd(R)

ranks = range(1, 11)
frob_errors = []
analytical_errors = []

for k in ranks:
    # Rank-k approximation
    R_k = (U_r[:, :k] * s_r[:k]) @ Vt_r[:k, :]
    frob_err = np.linalg.norm(R - R_k, 'fro')
    frob_errors.append(frob_err)
    
    # Analytical error: sqrt(sum of squared tail singular values)
    if k < len(s_r):
        anal_err = np.sqrt(np.sum(s_r[k:] ** 2))
    else:
        anal_err = 0.0
    analytical_errors.append(anal_err)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: Frobenius error vs rank
ax = axes[0]
ax.plot(ranks, frob_errors, 'o-', color='steelblue', lw=2, label='Computed ‖A - Â(k)‖_F')
ax.plot(ranks, analytical_errors, 's--', color='crimson', lw=1.5, label='Analytical √(Σσᵢ²)')
ax.set_xlabel('Rank k')
ax.set_ylabel('Frobenius error')
ax.set_title('Frobenius error vs rank k')
ax.legend()
ax.set_xticks(list(ranks))
ax.grid(True, alpha=0.3)

# Right: singular values (energy spectrum)
ax = axes[1]
ax.bar(list(ranks), s_r, color='steelblue', alpha=0.7)
ax.set_xlabel('Component i')
ax.set_ylabel('Singular value σᵢ')
ax.set_title('Singular value spectrum')
ax.set_xticks(list(ranks))
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Low-rank approximation of a 10×10 random matrix', fontsize=13)
plt.tight_layout()
plt.show()

print('Rank  | Frobenius error | Analytical error | Match')
print('-' * 60)
for k, fe, ae in zip(ranks, frob_errors, analytical_errors):
    print(f'  k={k:2d} | {fe:14.6f}  | {ae:15.6f}  | {np.isclose(fe, ae)}')

---
## 7. Image Compression Demo

A grayscale image is an $m \times n$ matrix. Its rank-$k$ SVD approximation uses only $k(m+n+1)$ numbers instead of $mn$. The compression ratio improves with smaller $k$.

In [ ]:
# --- Create a synthetic image with structure ---
n_rows, n_cols = 64, 64
xx, yy = np.meshgrid(np.linspace(-3, 3, n_cols), np.linspace(-3, 3, n_rows))

# Structured image: combination of Gaussians and a sine pattern
img = (np.exp(-(xx**2 + yy**2) / 2)
       + 0.5 * np.exp(-((xx-1.5)**2 + (yy-1.5)**2) / 0.5)
       + 0.3 * np.sin(2 * xx) * np.cos(2 * yy))

# Normalize to [0, 1]
img = (img - img.min()) / (img.max() - img.min())

# SVD of the image
U_img, s_img, Vt_img = np.linalg.svd(img, full_matrices=False)

def reconstruct(U, s, Vt, k):
    return (U[:, :k] * s[:k]) @ Vt[:k, :]

ks = [1, 5, 10, 20, n_rows]  # ranks to show (last = full)

fig, axes = plt.subplots(1, len(ks) + 1, figsize=(15, 3.5))

# Original
axes[0].imshow(img, cmap='viridis', vmin=0, vmax=1)
axes[0].set_title('Original\n(rank 64)')
axes[0].axis('off')

for ax, k in zip(axes[1:], ks):
    recon = reconstruct(U_img, s_img, Vt_img, k)
    recon = np.clip(recon, 0, 1)
    err   = np.linalg.norm(img - recon, 'fro')
    # Storage: k * (n_rows + n_cols + 1) numbers
    storage_ratio = (k * (n_rows + n_cols + 1)) / (n_rows * n_cols)
    ax.imshow(recon, cmap='viridis', vmin=0, vmax=1)
    ax.set_title(f'k = {k}\nerr={err:.2f}, {100*storage_ratio:.0f}% storage')
    ax.axis('off')

plt.suptitle('Rank-k SVD image reconstruction (64×64 synthetic image)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# --- Cumulative energy captured vs rank ---
cumulative_energy = np.cumsum(s_img ** 2) / np.sum(s_img ** 2)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(s_img) + 1), cumulative_energy * 100, 'o-', markersize=3,
        color='steelblue', lw=1.5)
ax.axhline(90, color='orange', ls='--', lw=1, label='90% energy')
ax.axhline(99, color='crimson', ls='--', lw=1, label='99% energy')

# Mark where thresholds are crossed
for threshold, color in [(90, 'orange'), (99, 'crimson')]:
    k_thresh = np.argmax(cumulative_energy >= threshold / 100) + 1
    ax.axvline(k_thresh, color=color, ls=':', lw=1)
    ax.text(k_thresh + 0.5, threshold - 4, f'k={k_thresh}', color=color, fontsize=9)

ax.set_xlabel('Rank k')
ax.set_ylabel('Cumulative energy captured (%)')
ax.set_title('Cumulative variance (energy) captured by top-k singular values')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Summary

| Decomposition | Exists for | Key Use |
|---------------|------------|----------|
| Cholesky $A = LL^\top$ | SPD matrices only | Stable solves, Gaussian sampling |
| Eigendecomposition $A = PDP^{-1}$ | Square, non-defective | Powers, Markov chains, spectral analysis |
| SVD $A = U\Sigma V^\top$ | **Any** matrix | Compression, PCA, pseudoinverse, geometry |

**The SVD is the fundamental theorem of linear algebra** — it subsumes all other decompositions and reveals the intrinsic geometry of any linear map.